# Dialeng Shell Integration Guide

This guide provides a complete overview of shell command execution in Dialeng, combining pshnb for execution and safecmd for security.

## Architecture Overview

```
+-------------------------------------------------------------------+
|                           DIALENG                                  |
+-------------------------------------------------------------------+
|  Code Cell                      Shell Cell                        |
|  +---------------+             +---------------+                  |
|  | Python code   |             | Bash commands |                  |
|  | + %bash magic |             | (fresh shell  |                  |
|  +-------+-------+             |  per cell)    |                  |
|          |                     +-------+-------+                  |
|          v                             |                          |
|  +---------------------------------------+                        |
|  |      Python Kernel (execnb/CaptureShell)                      |
|  |      + pshnb extension loaded                                 |
|  |      + @{var} expansion from Python namespace                 |
|  +---------------------------------------+                        |
|                    |                                              |
|                    v (if safe_mode=True)                          |
|  +---------------------------------------+                        |
|  |    safecmd validation layer           |                        |
|  |    - Allowlist-based command checking |                        |
|  |    - Raises DisallowedCmd on violation|                        |
|  +---------------------------------------+                        |
+-------------------------------------------------------------------+
```

## Three Ways to Run Shell Commands

### 1. `!command` Prefix (Recommended for one-liners)

Best for commands with flags like `-la`, `--verbose`, etc.

### 2. `%bash` Line Magic

Works for simple commands without flags. Supports variable expansion.

### 3. `%%bash` Cell Magic

Best for multi-line scripts and commands with flags.

In [ ]:
# Define a Python variable
project_name = "dialeng"

# Use it in a shell command (single quotes avoid history expansion)
%bash echo 'Working on: @{project_name}'

### 4. Dedicated Shell Cells

If enabled in Settings, create a shell cell from the **+ Shell** button:
- Bash syntax highlighting
- No Python kernel overhead
- Fresh shell session per cell

## Quick Start Examples

### Basic Commands

In [ ]:
# System info - use ! for commands with flags
!uname -a

In [ ]:
%%bash
# File operations - use %%bash for pipelines
ls -la *.py 2>/dev/null | head -5 || echo "No .py files"

In [ ]:
# Git status - use ! for flags
!git status --short 2>/dev/null || echo "Not a git repo"

### Variable Expansion

In [ ]:
# Python variables
pattern = "def.*:"
max_results = 10

In [ ]:
%%bash
# Use in shell command with %%bash for flags
grep -E "@{pattern}" *.py 2>/dev/null | head -@{max_results} || echo "No matches"

### Multi-line Scripts

In [ ]:
%%bash
echo "=== Project Structure ==="
echo ""
for dir in . notebooks docs services ui; do
    if [ -d "$dir" ]; then
        count=$(ls -1 "$dir" 2>/dev/null | wc -l)
        echo "$dir/: $count items"
    fi
done

## Safe Mode

Safe Mode validates shell commands against an allowlist before execution.

### Enabling Safe Mode

1. Find the **Safe** checkbox in the notebook toolbar
2. Check it to enable validation
3. All shell commands will be validated

### What Happens When Enabled

- **Allowed commands** execute normally
- **Blocked commands** raise `DisallowedCmd` with an explanation
- The "Safe" badge appears in shell cell headers

In [ ]:
%%bash
# This is allowed in Safe Mode
ls -la | head -5

In [ ]:
# This would be BLOCKED in Safe Mode
# Uncomment with Safe Mode enabled to see the error
# %%bash
# rm -rf /tmp/test

## Use Cases

### 1. Data Science Workflow

Combine Python data processing with shell tools:

In [ ]:
# Analyze data with Python
import json
data_files = ["config.json", "dialeng_config.json"]
print(f"Will check {len(data_files)} files")

In [ ]:
%%bash
# Use shell for quick file inspection
cat dialeng_config.json 2>/dev/null | head -20 || echo "File not found"

### 2. DevOps Tasks

In [ ]:
%%bash
# Check git status and recent commits
echo "=== Git Status ==="
git status --short 2>/dev/null | head -10 || echo "Not a git repo"
echo ""
echo "=== Recent Commits ==="
git --no-pager log --oneline -5 2>/dev/null || echo "No git history"

### 3. LLM-Assisted Development

When an LLM generates shell commands, Safe Mode provides protection:

In [ ]:
# LLM-generated command (with Safe Mode, dangerous commands are blocked)
llm_command = "find . -name '*.py' -type f | head -5"
print(f"LLM suggested: {llm_command}")

In [ ]:
%%bash
# Safe to execute (find without -exec is allowed)
find . -name '*.py' -type f 2>/dev/null | head -5 || echo "No .py files found"

## Syntax Quick Reference

| Syntax | When to Use | Example |
|--------|-------------|----------|
| `!cmd` | One-liners with flags | `!ls -la` |
| `%bash cmd` | Simple commands, no flags | `%bash pwd` |
| `%%bash` | Multi-line, any flags | `%%bash` + `ls -la` |

**Important:** `%bash` uses argparse, so command flags like `-la` are misinterpreted.

## Feature Comparison

| Feature | `!cmd` | `%bash` | `%%bash` | Shell Cell |
|---------|--------|---------|----------|------------|
| Commands with flags | Yes | No | Yes | Yes |
| Multi-line | No | No | Yes | Yes |
| @{var} expansion | No | Yes | Yes | Yes |
| Persistent state | No | Yes | Yes | No |
| Safe Mode | No | Yes | Yes | Yes |

## Troubleshooting

### `UsageError: unrecognized arguments: -la`
**Cause:** Using `%bash ls -la` - argparse sees `-la` as a magic flag
**Fix:** Use `!ls -la` or `%%bash` instead

### `bash: !": event not found`
**Cause:** Bash history expansion with `!` in double quotes
**Fix:** Use single quotes: `echo 'Hello!'` instead of `echo "Hello!"`

### `TIMEOUT` error with git commands
**Cause:** Git opens a pager (`less`) for output, which blocks waiting for input
**Fix:** Use `git --no-pager log` instead of `git log`

### Safe Mode checkbox is disabled
**Cause:** `shfmt` binary not installed
**Fix:** Install shfmt (`brew install shfmt` on macOS)

### Command blocked unexpectedly
**Cause:** Command not in allowlist
**Fix:** Disable Safe Mode for this notebook, or add to custom allowlist

### Variable expansion not working
**Cause:** Variable not defined in Python namespace
**Fix:** Ensure variable is defined before the shell command

## Related Documentation

- `pshnb_guide.ipynb` - Detailed pshnb usage
- `safecmd_guide.ipynb` - Safe Mode and security details
- [pshnb on GitHub](https://github.com/AnswerDotAI/pshnb)
- [safecmd on GitHub](https://github.com/AnswerDotAI/safecmd)